# Modeling: predicting long subway delays (≥ 5 min)

Input: `data/processed/delays_clean.csv`, produced by `01_eda.ipynb`.

In [1]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("../data/processed/delays_clean.csv", parse_dates=["datetime"])
print(df.shape)
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())

(45454, 13)
Date range: 2025-01-01 00:27:00 to 2026-08-31 23:59:00


In [2]:
# Time-based split: train on the earliest 80% of incidents, test on the latest 20%.
# A random split would let the model "see the future" during training.
df = df.sort_values("datetime").reset_index(drop=True)
cutoff = df["datetime"].quantile(0.8)

train = df[df["datetime"] < cutoff]
test = df[df["datetime"] >= cutoff]

for name, part in [("Train", train), ("Test", test)]:
    print(f"{name}: {len(part):,} incidents, "
          f"{part['datetime'].min():%Y-%m-%d} to {part['datetime'].max():%Y-%m-%d}, "
          f"long-delay share {part['long_delay'].mean():.1%}")

Train: 36,363 incidents, 2025-01-01 to 2026-05-12, long-delay share 18.9%
Test: 9,091 incidents, 2026-05-12 to 2026-08-31, long-delay share 14.3%


In [3]:
FEATURES = [
    "station_clean", "line_clean", "code_feature", "cause_group",
    "time_period", "hour", "weekday", "month", "is_weekend", "is_rush_hour",
]
TARGET = "long_delay"

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

In [4]:
# Baseline: always predict the most common class (not a long delay)
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")
print(confusion_matrix(y_test, y_pred))

Accuracy:  0.857
Precision: 0.000
Recall:    0.000
F1:        0.000
[[7795    0]
 [1296    0]]


Split and baseline:
- Data covers Jan 2025 to Aug 2026. Train: Jan 2025 - May 12, 2026 (36,363 incidents, 18.9% long);
  test: May 12 - Aug 31, 2026 (9,091 incidents, 14.3% long).
- The long-delay share is lower in the test period (possible seasonality or trend), so the test set
  is not distributed exactly like the training set.
- Majority-class baseline: accuracy 0.857 but precision, recall and F1 are all 0. Accuracy is
  misleading here, so models are compared mainly on F1 and recall.

In [5]:
# Long-delay share by month: is the lower share in the test period seasonal or a trend?
monthly = df.groupby(df["datetime"].dt.to_period("M"))["long_delay"].agg(["mean", "count"])
monthly["mean"] = monthly["mean"].round(3)
monthly

,mean,count
datetime,,
2025-01,0.218,2106
2025-02,0.203,2274
2025-03,0.232,1986
2025-04,0.198,2117
2025-05,0.184,2174
2025-06,0.209,1939
2025-07,0.176,2121
2025-08,0.141,2718
2025-09,0.168,1811


Monthly long-delay share shows both seasonality and a shift in 2026:
- Seasonal: higher in winter (20-23% in Jan-Mar 2025, 21.8% in Jan 2026), lowest in late summer (14.1% in Aug 2025).
- Shift: summer 2026 is lower than summer 2025 (e.g. June 14.3% vs 20.9%), while the number of
  recorded incidents per month is higher in 2026. One possible explanation is that more minor
  incidents are being recorded; the data alone cannot confirm this.
- Since the model learns from 2025 patterns, it may over-predict long delays in the 2026 test period.

In [6]:
results = []

def evaluate(name, y_true, y_pred):
    """Print and store the main metrics for one model."""
    row = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }
    results.append(row)
    print(pd.Series(row).drop("model").round(3).to_string())
    print(confusion_matrix(y_true, y_pred))

evaluate("Majority baseline", y_test, baseline.predict(X_test))

accuracy     0.857
precision      0.0
recall         0.0
f1             0.0
[[7795    0]
 [1296    0]]


In [7]:
# Categorical features are one-hot encoded; the two 0/1 flags are used as they are.
# Hour is left out: time_period already captures the non-linear hourly pattern.
CATEGORICAL = ["station_clean", "line_clean", "code_feature", "cause_group",
               "time_period", "weekday", "month"]
BINARY = ["is_weekend", "is_rush_hour"]

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("bin", "passthrough", BINARY),
])

for name, weights in [("Logistic regression", None),
                      ("Logistic regression (balanced)", "balanced")]:
    model = Pipeline([
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(max_iter=1000, class_weight=weights)),
    ])
    model.fit(X_train, y_train)
    print(f"--- {name} ---")
    evaluate(name, y_test, model.predict(X_test))
    print()

--- Logistic regression ---
accuracy     0.862
precision    0.526
recall         0.3
f1           0.382
[[7444  351]
 [ 907  389]]

--- Logistic regression (balanced) ---
accuracy     0.709
precision    0.305
recall       0.814
f1           0.444
[[5393 2402]
 [ 241 1055]]



In [8]:
pd.DataFrame(results).set_index("model").round(3)

,accuracy,precision,recall,f1
model,,,,
Majority baseline,0.857,0.000,0.000,0.000
Logistic regression,0.862,0.526,0.300,0.382
Logistic regression (balanced),0.709,0.305,0.814,0.444


Logistic regression results (test period May-Aug 2026):
- Both versions clearly beat the baseline (F1 0.38 and 0.44 vs 0), so the features carry real signal.
- Default weights: precision 0.53, recall 0.30. It is conservative and misses most long delays.
- Balanced class weights: recall 0.81, precision 0.31, best F1 (0.44). It catches most long delays
  at the cost of many false alarms.
- The choice depends on the use case: an early-warning tool would favour recall.
- Accuracy is lowest for the model with the best F1, confirming it is not a useful metric here.